In [ ]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_dataset
from core.logger import calculate_normalization_stats, print_data_summary
from core.training.sampler import create_kde_sampler
from core.data.dataset import create_dataloaders
from core.model.bert import BertForMaskedModeling
from core.training.pretrainer import setup_training
from torch.utils.data import random_split


In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BERT Configuration")

PyTorch version: 2.8.0+cu126
Using device: cuda
Working directory: /home/jessiez/osu_corpora

--- Loaded BERT Configuration ---
data:
  db_path: ./data/beatmap_dataset_test/
  max_seq_len: 1023
  val_split: 0.1
  chunk_size: 1000
training:
  batch_size: 8
  num_epochs: 5
  learning_rate: 0.0005
  weight_decay: 0.05
  warmup_ratio: 0.05
  min_lr: 1.0e-06
  use_amp: true
  checkpoint_dir: ./checkpoints/bert
  grad_clip_norm: 1.0
  sampling:
    method: kde
    kde_bandwidth: 0.5
    num_bins: 200
    expand_for_augmentation: true
fine_tuning:
  enabled: false
  checkpoint_path: ./checkpoints/bert/checkpoint_epoch_4.pth
  freeze_base_model: false
  learning_rate: 1.0e-05
  num_epochs: 3
  difficulty_weight: 0.4
  collection_label_weight: 0.4
  user_tag_weight: 0.2
  batching:
    batch_size: 8
    mix_difficulty: true
    mix_collection_labels: true
    mix_user_tags: false
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
components:
  use_rope: tr

In [ ]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['data']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_ratings = load_dataset(
    DATASET_PATH, 
    max_seq_len=config['data']['max_seq_len']
)

print_data_summary(all_beatmaps_data)

Using database: ./data/beatmap_dataset_test/
Loading raw data from Parquet dataset...
Loaded 2927 beatmaps and 2300556 hit objects.
Engineering features for all beatmaps (vectorized)...
Converting processed dataframes to tensors...


100%|██████████| 2926/2926 [00:00<00:00, 18975.76it/s]


Applying log transforms and filtering by sequence length...


100%|██████████| 2926/2926 [00:00<00:00, 6136.25it/s]

Finished loading and processing all data.

--- Data Summary ---
Total beatmaps: 2926
Vector dimension: 15
Metadata dimension: 6
Sequence length - Min: 48, Max: 1023, Avg: 697.3
--------------------


In [4]:
val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

print(f"Data split: {len(train_data)} training, {len(val_data)} validation")

train_data_list = [train_data.dataset[i] for i in train_data.indices]
val_data_list = [val_data.dataset[i] for i in val_data.indices]

train_difficulty_ratings = difficulty_ratings[train_data.indices]

sampler = create_kde_sampler(
    train_difficulty_ratings,
    bandwidth=config['training']['sampling']['kde_bandwidth'],
    expand_for_augmentation=config['training']['sampling']['expand_for_augmentation'],
    num_bins=config['training']['sampling'].get('num_bins', 100),
)

normalizer = calculate_normalization_stats(
    train_data_list,
    include_augmentation=config['training']['sampling']['expand_for_augmentation']
)

vector_stats = normalizer.get_vector_stats()
meta_stats = normalizer.get_metadata_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")
print(f"Metadata normalization stats for {len(meta_stats)} fields")

Data split: 2634 training, 292 validation
Creating optimized KDE sampler with bandwidth=0.5, bins=200...
KDE sampling - Min weight: 0.4258, Max weight: 17.1190
Calculating normalization statistics...
Including data augmentation in normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
----------------------------------------------------------------------
Field Name           Type         Param 1      Param 2      Description
----------------------------------------------------------------------
distance_diff        log+norm     4.4404       1.5274       Distance to previous hit object in pixels
cos_angle            mean/std     -0.0000      0.6959       Cosine of angle formed with previous two hit objects
sin_angle            mean/std     0.0000       0.6613       Sine of angle formed with previous two hit objects
velocity             log+norm     0.6866       0.3901       Velocity to previous hit object (pixels/ms)
cos_inner_angle      mean/s

In [5]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['training']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}, meta={sample_batch[2].shape}")

Created dataloaders with batch size: 8
Sample batch shapes: vectors=torch.Size([8, 1023, 15]), mask=torch.Size([8, 1023]), meta=torch.Size([8, 6])


In [6]:
model = BertForMaskedModeling.from_config(config, device)
model.print_info()

print("\nRunning a test forward pass with mixed precision (autocast)...")
with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        sample_vectors, sample_mask, sample_metadata = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)
        sample_metadata = sample_metadata.to(device)

        predictions, targets, _ = model(sample_vectors, sample_metadata, sample_mask)

print("\nBERT model created and tested successfully!")

Compiling BERT model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 25.44M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
Flash Attention: True
------------------------------

--- MLM Head Information ---
Task: Masked Modeling
Masking Ratio: 0.25
Model Compiled: True
------------------------------

Running a test forward pass with mixed precision (autocast)...


W0922 21:53:11.961000 440683 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1436] [3/0_1] Not enough SMs to use max_autotune_gemm mode



BERT model created and tested successfully!


In [7]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1 
        print(f"Loaded checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting training from scratch")

print(f"Training setup complete. Starting from epoch {start_epoch + 1}")
print(f"Total epochs: {config['training']['num_epochs']}")

Trainer initialized - AMP: True, Device: cuda
Training setup complete. Starting from epoch 1
Total epochs: 5


In [ ]:
print("\nStarting BERT training...")
print(f"BERT Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")
print(f"Components: RoPE={config['components'].get('use_rope', False)}")

if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True):
    effective_train_size = len(train_data) * 4 if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True) else len(train_data)
    print(f"Training samples: {len(train_data)} base maps -> {effective_train_size} with augmentation")
else:
    print(f"Training samples: {len(train_data)} base maps without augmentation")

metrics_tracker = trainer.train(start_epoch)

print("\nBERT training completed!")


Starting BERT training...
BERT Model: 6 layers, 512 dimensions
Components: RoPE=True
Training samples: 2634 base maps -> 10536 with augmentation

--- Starting Training ---
Epochs: 1 to 5
Batch Size: 8
Learning Rate: 0.0005
------------------------------------------------------------


Epoch 1 [Train]:   0%|          | 0/1317 [00:00<?, ?it/s]